# Notebook 07 — XGBoost i LightGBM

**Projekt:** Współzależność rynków aktywów cyfrowych i tradycyjnych systemów finansowych  

---

## Cel notebooka

Trenujemy dwa modele gradientowego boostingu na płaskich (nie sekwencyjnych) cechach:

- **XGBoost** — Extreme Gradient Boosting (standard w finansach)
- **LightGBM** — Microsoft LGBM, szybszy i często dokładniejszy na dużych danych

Modele drzewiaste:
1. Są naturalne odporne na skalowanie cech
2. Obsługują brakujące wartości
3. Dają bezpośrednią ważność cech (gain)
4. Świetnie działają z nierównowagą klas (scale_pos_weight)

## Optymalizacja hiperparametrów
Używamy Optuna (Bayesian optimization) — 50 prób dla każdego modelu.

In [1]:
# ============================================================
# Konfiguracja wspolna
# ============================================================
import os, warnings
os.environ.setdefault('KERAS_BACKEND', 'torch')
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
warnings.filterwarnings('ignore')
import importlib
import pandas as pd, numpy as np
import pipeline_lib as pl, models_lib as ml, run_analysis as ra
for m in (pl, ml, ra): importlib.reload(m)
pd.set_option('display.width', 160)
print(f'Ziarno losowe: {pl.SEED} | horyzont: {pl.HORIZON} sesji | embargo: {pl.EMBARGO} sesji')


Ziarno losowe: 42 | horyzont: 5 sesji | embargo: 10 sesji


In [2]:
# ============================================================
# Modele gradient boosting: waznosc cech
# ============================================================
xgb_imp, lgb_imp = ra.tree_importance_report()
print('XGBoost — 10 najwazniejszych cech:')
print(xgb_imp.head(10).to_string(index=False))
print('\nLightGBM — 10 najwazniejszych cech:')
print(lgb_imp.head(10).to_string(index=False))


XGBoost — 10 najwazniejszych cech:
               Cecha  Ważność
wig_realized_vol_21d 0.272966
           macro_vix 0.036996
macro_dxy_log_return 0.036824
  cross_corr_vol_30d 0.036689
btc_realized_vol_30d 0.034514
btc_rolling_kurt_30d 0.033093
btc_rolling_skew_30d 0.032606
      macro_vix_ma30 0.030649
     btc_amihud_ma30 0.030362
     wig_amihud_ma30 0.029786

LightGBM — 10 najwazniejszych cech:
                Cecha  Ważność
   cross_corr_vol_30d      227
 wig_realized_vol_21d      162
            macro_vix      152
      wig_amihud_ma30      107
       macro_vix_ma30       97
            macro_dxy       94
wig_volume_normalized       83
           btc_rsi_14       82
 btc_rolling_kurt_30d       81
 btc_realized_vol_30d       81


In [3]:
# ============================================================
# Wyniki pozaprobkowe modeli drzewiastych
# ============================================================
oof = pd.read_csv('data/processed/oof_walkforward.csv', index_col=0, parse_dates=True)
y = oof['y_true'].astype(int); common = oof.dropna(subset=['p_ens']).index
rows = []
for name, col in [('XGBoost','p_xgb'), ('LightGBM','p_lgb')]:
    s = oof.loc[common, col].dropna(); yy = y.reindex(s.index).values
    half = len(s)//2
    thr = pl.select_threshold(yy[:half], s.values[:half]) if yy[:half].sum() else 0.5
    m = pl.evaluate(yy, s.values, thr); m['Model'] = name; rows.append(m)
trees = pd.DataFrame(rows).set_index('Model')
trees.to_csv('results/tables/07_xgb_lgb_comparison.csv')
print(trees[['auc_roc','auc_pr','auc_pr_lift','f1','precision','recall','brier']].to_string())


          auc_roc  auc_pr  auc_pr_lift      f1  precision  recall   brier
Model                                                                    
XGBoost    0.8326  0.4124       18.218  0.3478     0.2759  0.4706  0.0199
LightGBM   0.8103  0.4385       19.371  0.3265     0.2500  0.4706  0.0219
